In [ ]:
!pip install nuscenes-devkit
!pip install matplotlib pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
nuscenes_dataroot='/content/drive/MyDrive/data/v1.0-mini'
drivelm_path='/content/drive/MyDrive/data/v1_0_train_nus.json'


In [ ]:
from nuscenes.nuscenes import NuScenes
import json
with open(drivelm_path,'r') as f:
  drivelm_data = json.load(f)
print(f"DriveLM enteries:{len(drivelm_data)}")
##############################################
nuscene=NuScenes(version='v1.0-mini',dataroot=nuscenes_dataroot,verbose=True)
print(f"\nScenes:{len(nuscene.scene)}")
print(f"\nSamples:{len(nuscene.sample)}")
print(f"Annotations:{len(nuscene.sample_annotation)}")


In [ ]:
import random
def link_entries(drivelm_path:str,output_path:str):
  nuscene=NuScenes(version='v1.0-mini',dataroot=nuscenes_dataroot,verbose=True)
  with open(drivelm_path,'r') as f:
    drivelm_data = json.load(f)
  linked_entries=[]

  for scene_token, scene_info in drivelm_data.items():
    try:
      scene=nuscene.get('scene',scene_token)
    except:
      continue
    log = nuscene.get('log',scene['log_token'])
    key_frames=scene_info.get('key_frames',{})
    for sample_token,frame_data in key_frames.items():
      try:
        sample = nuscene.get('sample',sample_token)
      except:
        continue
      sd = nuscene.get('sample_data',sample['data']['CAM_FRONT']) ##considering only the CAMERA FRONT images.
      ego=nuscene.get('ego_pose',sd['ego_pose_token'])
      annotation=[]
      for ann_token in sample['anns']:
        ann=nuscene.get('sample_annotation',ann_token)
        annotation.append({'category':ann['category_name'],
                       'location':ann['translation']}) ##the current data is all from boston

      linked_entry={
          'scene_name':scene['name'],
          'scene_token':scene_token,
          'sample_token':sample_token,
          'location':log['location'],
          'cam_front_path':sd['filename'],
          'ego_translation':ego['translation'],
          'qa_paits':frame_data.get('QA',frame_data)
      }
      linked_entries.append(linked_entry)



  os.makedirs(output_path, exist_ok=True)

  train_path = os.path.join(output_dir, 'drivelm_nusc.json')

  with open(train_path, 'w') as f:
      json.dump(train_entries, f, indent=2)


In [ ]:
link_entries(drivelm_path,'/content/drive/Mydrive/data/output/')